In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from tqdm import tqdm
import pandas as pd
import time
import numpy as np

##Import data
csvfile = 'marketing_campaign_cleaned_NODUMMIES'
df = pd.read_csv(csvfile)

#Synthcity
import sys
import warnings

warnings.filterwarnings("ignore")

import synthcity.logger as log
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

log.add(sink=sys.stderr, level="INFO")

#Fit GAN using ALL training data
from synthcity.plugins import Plugins

syn_model_full = Plugins().get("ctgan")

##########################FUNCTIONS#################################

def dummify_columns(df):
    # Dummify marital and educational
    dummify_marital = pd.get_dummies(df['Marital_Status'],prefix='marital')
    df = pd.concat([df, dummify_marital],axis=1)

    dummify_edu = pd.get_dummies(df['Education'],prefix='education')
    df = pd.concat([df, dummify_edu], axis=1)

    # Drop transformed cols
    df.drop(columns=['Marital_Status', 'Education'], inplace=True)
    
    return df

def train_and_evaluate(df_train, df_test):
    # Split into features and target for train and test datasets
    X_train = df_train.drop('Response', axis=1)
    y_train = df_train['Response']
    X_test = df_test.drop('Response', axis=1)
    y_test = df_test['Response']
    
    # Identify overlapping columns in train and test: note that the train set is only small (7.5p), columns may be missing
    # if we add only a little bit of synth data, there is a chance that same column is missing in synth data
    common_columns = set(X_train.columns).intersection(X_test.columns)
    
    # Keep only common columns in train and test
    X_train = X_train[common_columns]
    X_test = X_test[common_columns]
    

    # Train a Random Forest model
    model = RandomForestClassifier(n_estimators = 500, max_depth = None, 
                                   max_features = 'auto', criterion = 'gini', min_samples_split = 2)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store metrics
    results = {
        'accuracy': report['accuracy'],
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
        'auc-roc': roc_auc
    }

    return results





<stdin>:1:10: fatal error: 'omp.h' file not found
#include <omp.h>
         ^~~~~~~
1 error generated.


[KeOps] Warning : omp.h header is not in the path, disabling OpenMP.
[KeOps] Warning : Cuda libraries were not detected on the system ; using cpu only mode
2023-08-04 19:40:40,316 - Created a temporary directory at /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmpzgiqejr3
2023-08-04 19:40:40,318 - Writing /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmpzgiqejr3/_remote_module_non_scriptable.py


In [2]:
# OTHER extrinisc FASTER had randomstate = 4

# Define synthetic set sizes


countlen = len(df)*0.7*0.85

syn_sizes = [1, 0.5*countlen, 1*countlen,  3*countlen,  5*countlen, 
            8*countlen, 12*countlen, 18*countlen, 32*countlen, 48*countlen, 64*countlen]

n_iterations = 100  # Number of bootstrapping iterations
syn_model = Plugins().get('ctgan')

# Placeholder for the results
results = []

# Bootstrap iteration loop
for i in range(n_iterations):
    # Resample entire dataset

    start_time = time.time()
    
    # Set the size of your bootstrap sample. This could be the size of your original dataset
    bootstrap_size = int(0.7 * df.shape[0])

    # Perform bootstrapping
    df_train_main = resample(df, replace=True, n_samples=bootstrap_size, random_state=i*124)

    # Find the Out-of-Bag samples
    oob_index = df.index.difference(df_train_main.index)
    df_test = df.loc[oob_index]

    # Perform train/test split
    df_train_1, df_train_2 = train_test_split(df_train_main, test_size=0.15, random_state=i*1144)

    # Resample
    #df_test_resample = resample(df_test,replace=True)
    #df_train_1_resample = resample(df_train_1,replace=True)
    #df_train_2_resample = resample(df_train_2,replace=True)
    
    loader = GenericDataLoader(df_train_1, target_column='Response')
    syn_model.fit(loader)
    # Loop through the different synthetic set sizes
    for size in syn_sizes:
        # Generate synthetic set of the current size based on resampled training data
        #syn_set = syn_model.generate(count=size,random_state=i*124).dataframe()
        syn_set = syn_model.generate(count=size,random_state=i*114).dataframe()

        print(syn_set['Response'].mean() * 100)

        # Add synthetic data to resampled training data
        df_train_combined = pd.concat([df_train_2, syn_set], axis=0)

        # Dummify train and test datasets
        df_train_combined = dummify_columns(df_train_combined)
        df_test_dummified = dummify_columns(df_test.copy())  # .copy() to avoid SettingWithCopyWarning

        # Train and evaluate
        metrics = train_and_evaluate(df_train_combined, df_test_dummified)

        # Store the results with additional information
        metrics['syn_size'] = size
        metrics['iteration'] = i
        results.append(metrics)

    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Time: {execution_time} seconds")
    print(f"Iteration: {i} " )
    
# Convert to DataFrame
results_exc_df_1 = pd.DataFrame(results)

 32%|████████████▉                           | 649/2000 [06:45<14:04,  1.60it/s]


0.0
10.076335877862595
10.526315789473683
10.038119440914867
10.443665192864767
10.643163411148166
10.334751953249063
10.895617192462417
11.002072267346307
10.7885794137263
11.005645142081319
Time: 708.2517490386963 seconds
Iteration: 0 


 42%|████████████████▉                       | 849/2000 [08:31<11:33,  1.66it/s]


0.0
11.145038167938932
12.967200610221205
11.257941550190598
12.1207501143467
12.28203906622201
11.725846407927333
11.0734702519585
11.516566229188005
11.401530790484962
11.38794273872758
Time: 824.5807616710663 seconds
Iteration: 1 


 32%|████████████▉                           | 649/2000 [06:10<12:50,  1.75it/s]


...

0.0
15.877862595419847
17.162471395881006
17.458703939008892
17.51791431620674
17.627441638875656
17.893667026615002
17.97586279906839
17.70240335373842
17.705719820878457
17.467784579472646
Time: 704.0144896507263 seconds
Iteration: 97 


 32%|████████████▉                           | 649/2000 [05:24<11:14,  2.00it/s]


0.0
14.351145038167939
17.31502669717773
15.65438373570521
15.962799207196218
16.179132920438306
16.73124563297974
16.362481473639637
16.354238620393968
16.325785244704164
16.26015291903866
Time: 620.6505670547485 seconds
Iteration: 98 


 65%|█████████████████████████▎             | 1299/2000 [10:28<05:38,  2.07it/s]


0.0
17.251908396946565
16.323417238749048
16.11181702668361
16.19149260558012
15.817055740828966
15.295686971987548
15.31230150328181
15.537241264321272
15.757296662114523
15.507467308196174
Time: 901.2996318340302 seconds
Iteration: 99 


In [ ]:
results_exc_df_1.to_clipboard()